# Chapitre 4 - Lab : évolution de schéma dans PostgreSQL

Ce lab confronte le cours 4 (flux de données par les bases) à un vrai moteur,
PostgreSQL 16. Nous allons :

1. ajouter une colonne **sans réécrire** la table, et voir où PostgreSQL range la valeur
   par défaut ;
2. observer **physiquement** des lignes de plusieurs époques dans les mêmes pages ;
3. comparer avec les changements qui **réécrivent** toute la table ;
4. mesurer le piège des verrous d'un `ALTER TABLE` en production ;
5. faire évoluer des documents JSONB sans effacer les champs inconnus ;
6. comparer les formats texte et binaire du protocole de PostgreSQL.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/encoding/compose.yml -p ddia_encoding up -d     # démarrer
docker compose -f docker/encoding/compose.yml -p ddia_encoding down -v   # arrêter et effacer
```

PostgreSQL écoute sur le port **10532** de la machine (utilisateur, mot de passe et base :
`ddia` / `ddia` / `evolution`).

In [1]:
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg

from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 100)

DSN = "host=localhost port=10532 user=ddia password=ddia dbname=evolution"
con = None
for essai in range(30):
    try:
        con = psycopg.connect(DSN, autocommit=True)
        break
    except psycopg.OperationalError:
        time.sleep(1)
assert con is not None, "PostgreSQL ne répond pas : avez-vous lancé docker compose up ?"
print(con.execute("SELECT version()").fetchone()[0][:60])
con.execute("CREATE EXTENSION IF NOT EXISTS pageinspect")

def q(sql, params=None, c=None):
    return (c or con).execute(sql, params).fetchall()

def fichier(table):
    """Fichier physique (relfilenode) et taille de la table."""
    return q("SELECT pg_relation_filenode(%s::regclass), pg_relation_size(%s::regclass)", (table, table))[0]

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linu


## 1. Ajouter une colonne sans réécrire (section *Different values written at different times*)

Créons une table d'un million de clients, puis ajoutons une colonne `pays` avec une valeur
par défaut constante. Surveillons le **fichier** de la table (son numéro `relfilenode`) et
sa taille : si PostgreSQL réécrivait la table, il créerait un nouveau fichier.

In [2]:
con.execute("DROP TABLE IF EXISTS clients")
con.execute("CREATE TABLE clients (id int PRIMARY KEY, nom text NOT NULL, solde numeric(12, 2))")
con.execute("""INSERT INTO clients SELECT i, 'client' || i, round((random() * 1000)::numeric, 2)
               FROM generate_series(1, 1000000) AS i""")
con.execute("VACUUM ANALYZE clients")
avant = fichier("clients")
t = time.perf_counter()
con.execute("ALTER TABLE clients ADD COLUMN pays text DEFAULT 'FR'")
duree_add = time.perf_counter() - t
apres = fichier("clients")
print(f"ADD COLUMN ... DEFAULT 'FR' : {duree_add * 1e3:.1f} ms")
print(f"fichier avant : {avant[0]} ({avant[1] / 1e6:.1f} Mo) ; après : {apres[0]} ({apres[1] / 1e6:.1f} Mo)")
pd.DataFrame(q("""SELECT attname, atthasmissing, attmissingval FROM pg_attribute
                  WHERE attrelid = 'clients'::regclass AND attnum > 0 ORDER BY attnum"""),
             columns=["colonne", "valeur manquante gérée ?", "valeur fournie à la lecture"])

ADD COLUMN ... DEFAULT 'FR' : 0.4 ms
fichier avant : 16800 (52.2 Mo) ; après : 16800 (52.2 Mo)


,colonne,valeur manquante gérée ?,valeur fournie à la lecture
0,id,False,NaN
1,nom,False,NaN
2,solde,False,NaN
3,pays,True,{FR}


Quelques millisecondes, même fichier, même taille : la table n'a **pas** été réécrite.
PostgreSQL (depuis la version 11) range la valeur par défaut dans le catalogue
(`pg_attribute.attmissingval`) et la fournit à la lecture de toute ligne écrite avant
l'ajout de la colonne. C'est exactement le mécanisme décrit par le livre : la base
**paraît** encodée avec un seul schéma, alors que le stockage mélange plusieurs époques.

> **À retenir.** Ajouter une colonne avec un défaut constant ne touche qu'au catalogue ;
> les anciennes lignes restent telles quelles sur le disque et reçoivent le défaut à la
> lecture.